# Hannover Messe Survey Analysis - Modular Notebook

This notebook analyzes student participation and engagement at the Hannover Messe fair.

**Features:**
- Supports manual sample data (14 students from Hannover Messe)
- Supports Microsoft Forms CSV export import
- Modular functions for data loading, cleaning, analysis, and visualization
- Configurable column mapping for Microsoft Forms exports

**Analysis includes:**
- Attendance and networking behavior
- Job positions discovered
- Companies visited
- Career fields explored
- Overall specialization distribution

**Microsoft Forms Integration:**
- Use the standalone `forms_to_csv_adapter.py` script to convert Forms exports
- Or configure this notebook to load Forms CSV directly

## 1. Configuration

**Option A: Use the standalone adapter script (Recommended)**
```bash
python forms_to_csv_adapter.py your_forms_export.csv student_data_structured.csv
```
Then set `DATA_SOURCE = 'sample'` and the notebook will use the converted CSV.

**Option B: Load Microsoft Forms CSV directly in notebook**
1. Set `DATA_SOURCE = 'microsoft_forms'`
2. Update `FORMS_CSV_PATH` with your CSV filename
3. Adjust `FORMS_COLUMN_MAPPING` to match your form's column names
4. Run all cells

In [ ]:
# ============================================
# CONFIGURATION SECTION
# ============================================

# Data source: 'sample' or 'microsoft_forms'
DATA_SOURCE = 'sample'  # Change to 'microsoft_forms' to use CSV import

# Path to Microsoft Forms CSV export (only used if DATA_SOURCE = 'microsoft_forms')
FORMS_CSV_PATH = 'forms_export.csv'

# Column mapping for Microsoft Forms export
FORMS_COLUMN_MAPPING = {
    'Name': 'Student',
    'Did you attend Hannover Messe?': 'attended_messe',
    'Did you speak to company representatives?': 'spoke_to_people',
    'How many job positions did you discover?': 'positions_seen',
    'Which companies did you visit? (separate with commas)': 'companies_seen',
    'Which career fields interested you? (separate with commas)': 'career_fields_seen',
    'Additional notes': 'notes'
}

# Output CSV path for cleaned data
OUTPUT_CSV_PATH = 'student_data_structured.csv'

# Aggregate specialization data (from separate survey)
SPECIALIZATION_AGGREGATE = {
    'Full-stack': 7,
    'BI/DA': 3,
    'Missing/Both': 10
}

## 2. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
print("Libraries imported successfully")

## 3. Data Loading Functions

In [ ]:
def get_sample_data():
    """Return the 14-student Hannover Messe sample dataset."""
    messe_data = {
        "Blerina": {"attended_messe": False, "spoke_to_people": False, "positions_seen": 0, "companies_seen": [], "career_fields_seen": ["Data Analyzer"], "notes": "Did not attend but interested in Data Analyzer field"},
        "Menjung": {"attended_messe": True, "spoke_to_people": True, "positions_seen": 15, "companies_seen": ["Quantum"], "career_fields_seen": ["Fullstack"], "notes": None},
        "Hani": {"attended_messe": True, "spoke_to_people": True, "positions_seen": 7, "companies_seen": ["Siemens", "Nestle", "Saudi Arabia", "Adesso"], "career_fields_seen": ["Frontend/UX", "BI"], "notes": None},
        "Hadis": {"attended_messe": True, "spoke_to_people": True, "positions_seen": 3, "companies_seen": ["Siemens"], "career_fields_seen": ["Web Development"], "notes": None},
        "Yarema": {"attended_messe": True, "spoke_to_people": True, "positions_seen": 5, "companies_seen": ["Microsoft"], "career_fields_seen": ["Software Developer"], "notes": None},
        "Viktoriia": {"attended_messe": False, "spoke_to_people": False, "positions_seen": 0, "companies_seen": [], "career_fields_seen": ["Marketing", "Data Analytics"], "notes": "Did not attend but interested in Marketing and Data Analytics"},
        "Andi": {"attended_messe": False, "spoke_to_people": False, "positions_seen": 0, "companies_seen": [], "career_fields_seen": ["Technician", "System Administrator"], "notes": "Did not attend but interested in Technician and System Admin roles"},
        "Ihor": {"attended_messe": None, "spoke_to_people": None, "positions_seen": None, "companies_seen": ["Bundeswehr", "Siemens"], "career_fields_seen": ["System Administrator"], "notes": "Attendance/networking status unclear; visited companies listed"},
        "Svitlana": {"attended_messe": True, "spoke_to_people": True, "positions_seen": 7, "companies_seen": ["AWS", "Agile Robots", "Bosch", "Siemens", "Leibniz Uni"], "career_fields_seen": ["AI Engineer", "Data Science"], "notes": "AI and Data Science merged in analysis"},
        "Sofiia": {"attended_messe": True, "spoke_to_people": True, "positions_seen": 5, "companies_seen": ["AWS", "Microsoft", "Bundeswehr", "Food Court"], "career_fields_seen": ["FullStack"], "notes": None},
        "Kareem": {"attended_messe": True, "spoke_to_people": True, "positions_seen": 2, "companies_seen": ["SAP", "Microsoft", "Bundeswehr", "Lenovo"], "career_fields_seen": ["Frontend"], "notes": None},
        "Omar": {"attended_messe": True, "spoke_to_people": True, "positions_seen": 2, "companies_seen": ["Bundeswehr"], "career_fields_seen": [], "notes": "Career field not specified"},
        "Tetiana": {"attended_messe": True, "spoke_to_people": True, "positions_seen": None, "companies_seen": ["Bundeswehr", "BMW"], "career_fields_seen": ["Frontend"], "notes": "Position count not provided"},
        "Yevhen": {"attended_messe": None, "spoke_to_people": None, "positions_seen": None, "companies_seen": ["Samsung", "Bundeswehr", "BMW"], "career_fields_seen": ["FullStack"], "notes": "Attendance/networking/position count unclear; visited companies listed"}
    }
    df = pd.DataFrame.from_dict(messe_data, orient='index')
    df.index.name = 'Student'
    return df.reset_index()

def load_microsoft_forms_csv(filepath, column_mapping):
    """Load and map Microsoft Forms CSV export to standardized format."""
    try:
        df = pd.read_csv(filepath)
        print(f"Loaded {len(df)} rows from {filepath}")
        df = df.rename(columns=column_mapping)
        required_cols = ['Student', 'attended_messe', 'spoke_to_people', 'positions_seen', 'companies_seen', 'career_fields_seen']
        for col in required_cols:
            if col not in df.columns:
                print(f"Warning: Column '{col}' not found. Creating empty column.")
                df[col] = None
        if 'notes' not in df.columns:
            df['notes'] = None
        return df
    except FileNotFoundError:
        print(f"Error: File '{filepath}' not found.")
        return None
    except Exception as e:
        print(f"Error loading CSV: {e}")
        return None

print("Data loading functions defined")

## 4. Data Cleaning Functions

In [ ]:
def normalize_boolean(value):
    if pd.isna(value) or value == '' or value is None:
        return None
    if isinstance(value, bool):
        return value
    value_str = str(value).strip().lower()
    if value_str in ['yes', 'true', '1', 'ja', 'y']:
        return True
    elif value_str in ['no', 'false', '0', 'nein', 'n']:
        return False
    return None

def parse_list_field(value, separators=[',', ';', '/']):
    if pd.isna(value) or value == '' or value is None:
        return []
    if isinstance(value, list):
        return [str(item).strip() for item in value if str(item).strip()]
    value_str = str(value)
    for sep in separators:
        if sep in value_str:
            items = [item.strip() for item in value_str.split(sep)]
            return [item for item in items if item]
    cleaned = value_str.strip()
    return [cleaned] if cleaned else []

def normalize_numeric(value):
    if pd.isna(value) or value == '' or value is None:
        return None
    try:
        return int(float(value))
    except (ValueError, TypeError):
        return None

def clean_dataframe(df):
    df = df.copy()
    df['attended_messe'] = df['attended_messe'].apply(normalize_boolean)
    df['spoke_to_people'] = df['spoke_to_people'].apply(normalize_boolean)
    df['positions_seen'] = df['positions_seen'].apply(normalize_numeric)
    df['companies_seen'] = df['companies_seen'].apply(parse_list_field)
    df['career_fields_seen'] = df['career_fields_seen'].apply(parse_list_field)
    df['num_companies'] = df['companies_seen'].apply(len)
    df['num_career_fields'] = df['career_fields_seen'].apply(len)
    return df

print("Data cleaning functions defined")

## 5. Career Field Categorization

In [ ]:
def categorize_career_field(field):
    """Note: AI and Data Science are merged into 'Data/AI/BI' category."""
    if not field or pd.isna(field):
        return 'Other'
    field_lower = str(field).lower()
    if 'fullstack' in field_lower or 'full-stack' in field_lower:
        return 'FullStack'
    elif 'frontend' in field_lower or 'ux' in field_lower:
        return 'Frontend/UX'
    elif any(term in field_lower for term in ['data', 'ai', 'bi', 'analytics', 'analyst']):
        return 'Data/AI/BI'
    elif 'system' in field_lower or 'admin' in field_lower:
        return 'System Admin'
    elif 'web' in field_lower:
        return 'Web Development'
    elif 'software' in field_lower or 'developer' in field_lower:
        return 'Software Developer'
    elif 'marketing' in field_lower:
        return 'Marketing'
    elif 'technician' in field_lower:
        return 'Technician'
    return 'Other'

def aggregate_career_fields(df):
    career_field_categories = []
    for fields in df['career_fields_seen']:
        for field in fields:
            category = categorize_career_field(field)
            career_field_categories.append(category)
    return Counter(career_field_categories)

print("Career field categorization functions defined")

## 6. Load and Clean Data

In [ ]:
if DATA_SOURCE == 'sample':
    print("Loading sample data (14 students from Hannover Messe)...")
    df = get_sample_data()
    print(f"Loaded {len(df)} students")
elif DATA_SOURCE == 'microsoft_forms':
    print(f"Loading Microsoft Forms export from: {FORMS_CSV_PATH}")
    df = load_microsoft_forms_csv(FORMS_CSV_PATH, FORMS_COLUMN_MAPPING)
    if df is None:
        print("\nFalling back to sample data...")
        df = get_sample_data()
else:
    print(f"Unknown DATA_SOURCE: {DATA_SOURCE}. Using sample data.")
    df = get_sample_data()

print("\nCleaning data...")
df = clean_dataframe(df)
print("\nData preview:")
print(df[['Student', 'attended_messe', 'spoke_to_people', 'positions_seen', 'num_companies', 'num_career_fields']].head(10))

## 7. Summary Statistics

In [ ]:
def generate_summary_statistics(df):
    print("\n" + "="*50)
    print("HANNOVER MESSE SURVEY SUMMARY")
    print("="*50)
    print(f"\nTotal Students Surveyed: {len(df)}")
    attended_known = df['attended_messe'].notna()
    attended_yes = df['attended_messe'] == True
    print(f"\n--- Attendance ---")
    print(f"Confirmed Attendees: {attended_yes.sum()}")
    print(f"Did Not Attend: {(df['attended_messe'] == False).sum()}")
    print(f"Unknown/Not Reported: {(~attended_known).sum()}")
    if attended_known.sum() > 0:
        print(f"Attendance Rate (of known): {attended_yes.sum() / attended_known.sum() * 100:.1f}%")
    spoke_known = df['spoke_to_people'].notna()
    spoke_yes = df['spoke_to_people'] == True
    print(f"\n--- Networking ---")
    print(f"Spoke to People: {spoke_yes.sum()}")
    print(f"Did Not Network: {(df['spoke_to_people'] == False).sum()}")
    print(f"Unknown/Not Reported: {(~spoke_known).sum()}")
    if spoke_known.sum() > 0:
        print(f"Networking Rate (of known): {spoke_yes.sum() / spoke_known.sum() * 100:.1f}%")
    positions_data = df[df['positions_seen'].notna()]['positions_seen']
    if len(positions_data) > 0:
        print(f"\n--- Job Positions Discovered ---")
        print(f"Average Positions Seen: {positions_data.mean():.1f}")
        print(f"Median Positions Seen: {positions_data.median():.0f}")
        print(f"Range: {positions_data.min():.0f} - {positions_data.max():.0f}")
        print(f"Total Positions Discovered: {positions_data.sum():.0f}")
    print(f"\n--- Companies ---")
    print(f"Average Companies per Student: {df['num_companies'].mean():.1f}")
    print(f"Students Who Visited Companies: {(df['num_companies'] > 0).sum()}")
    return df

df = generate_summary_statistics(df)

## 8. Company & Career Field Analysis

In [ ]:
# Company Analysis
all_companies = []
for companies in df['companies_seen']:
    all_companies.extend(companies)
company_counts = Counter(all_companies)
print("\n" + "="*50)
print("MOST POPULAR COMPANIES AT HANNOVER MESSE")
print("="*50 + "\n")
for company, count in company_counts.most_common():
    print(f"{company}: {count} student(s)")

# Career Field Analysis
field_counts = aggregate_career_fields(df)
print("\n" + "="*50)
print("CAREER FIELDS EXPLORED AT HANNOVER MESSE")
print("="*50)
print("(Note: AI and Data Science merged into Data/AI/BI category)\n")
for field, count in field_counts.most_common():
    print(f"{field}: {count} mention(s)")

# Specialization Distribution
print("\n" + "="*50)
print("OVERALL CLASS SPECIALIZATION DISTRIBUTION")
print("="*50)
print("(From separate aggregate survey)\n")
total_responses = sum(SPECIALIZATION_AGGREGATE.values())
for spec, count in SPECIALIZATION_AGGREGATE.items():
    percentage = (count / total_responses) * 100
    print(f"{spec}: {count} students ({percentage:.1f}%)")
print(f"\nTotal: {total_responses} students")

## 9. Visualizations

In [ ]:
fig = plt.figure(figsize=(18, 12))
fig.suptitle('Hannover Messe Survey Analysis Dashboard', fontsize=16, fontweight='bold')
attended_known = df['attended_messe'].notna()
attended_yes = df['attended_messe'] == True
ax1 = plt.subplot(2, 3, 1)
labels = ['Attended', 'Did Not Attend', 'Unknown']
sizes = [attended_yes.sum(), (df['attended_messe'] == False).sum(), (~attended_known).sum()]
colors = ['#2ecc71', '#e74c3c', '#95a5a6']
ax1.pie(sizes, labels=labels, autopct='%1.1f%%', startangle=90, colors=colors)
ax1.set_title('Messe Attendance', fontweight='bold')
ax2 = plt.subplot(2, 3, 2)
positions_df = df[df['positions_seen'].notna()].sort_values('positions_seen', ascending=False)
if len(positions_df) > 0:
    ax2.barh(positions_df['Student'], positions_df['positions_seen'], color='steelblue')
    ax2.set_xlabel('Number of Positions')
    ax2.set_title('Job Positions Discovered per Student', fontweight='bold')
    ax2.grid(axis='x', alpha=0.3)
ax3 = plt.subplot(2, 3, 3)
if len(company_counts) > 0:
    top_companies = dict(company_counts.most_common(10))
    ax3.barh(list(top_companies.keys()), list(top_companies.values()), color='teal')
    ax3.set_xlabel('Number of Students')
    ax3.set_title('Top 10 Companies Visited', fontweight='bold')
    ax3.grid(axis='x', alpha=0.3)
ax4 = plt.subplot(2, 3, 4)
if len(field_counts) > 0:
    field_counts_sorted = dict(sorted(field_counts.items(), key=lambda x: x[1], reverse=True))
    ax4.bar(field_counts_sorted.keys(), field_counts_sorted.values(), color='coral')
    ax4.set_ylabel('Number of Mentions')
    ax4.set_title('Career Fields Explored\n(AI/Data merged)', fontweight='bold')
    ax4.tick_params(axis='x', rotation=45)
    ax4.grid(axis='y', alpha=0.3)
ax5 = plt.subplot(2, 3, 5)
spec_labels = list(SPECIALIZATION_AGGREGATE.keys())
spec_sizes = list(SPECIALIZATION_AGGREGATE.values())
spec_colors = ['#3498db', '#e67e22', '#95a5a6']
ax5.pie(spec_sizes, labels=spec_labels, autopct='%1.1f%%', startangle=90, colors=spec_colors)
ax5.set_title('Overall Class Specialization\n(Aggregate Survey)', fontweight='bold')
ax6 = plt.subplot(2, 3, 6)
scatter_data = df[(df['spoke_to_people'] == True) & (df['positions_seen'].notna())]
if len(scatter_data) > 0:
    ax6.scatter(scatter_data['num_companies'], scatter_data['positions_seen'], s=100, alpha=0.6, color='darkgreen', edgecolors='black', linewidth=1)
    ax6.set_xlabel('Number of Companies Visited')
    ax6.set_ylabel('Positions Discovered')
    ax6.set_title('Companies Visited vs Positions Found', fontweight='bold')
    ax6.grid(alpha=0.3)
    if len(scatter_data) > 2:
        z = np.polyfit(scatter_data['num_companies'], scatter_data['positions_seen'], 1)
        p = np.poly1d(z)
        x_trend = np.linspace(scatter_data['num_companies'].min(), scatter_data['num_companies'].max(), 100)
        ax6.plot(x_trend, p(x_trend), "r--", alpha=0.5, linewidth=2, label='Trend')
        ax6.legend()
plt.tight_layout()
plt.show()

## 10. Student Engagement Heatmap

In [ ]:
engagement_df = df[['Student', 'attended_messe', 'spoke_to_people', 'num_companies', 'positions_seen']].copy()
engagement_df['attended_messe'] = engagement_df['attended_messe'].map({True: 1, False: 0, None: 0.5})
engagement_df['spoke_to_people'] = engagement_df['spoke_to_people'].map({True: 1, False: 0, None: 0.5})
engagement_df['positions_seen'] = engagement_df['positions_seen'].fillna(0)
engagement_matrix = engagement_df.set_index('Student')[['attended_messe', 'spoke_to_people', 'num_companies', 'positions_seen']]
max_companies = engagement_matrix['num_companies'].max()
max_positions = engagement_matrix['positions_seen'].max()
if max_companies > 0:
    engagement_matrix['num_companies'] = engagement_matrix['num_companies'] / max_companies
if max_positions > 0:
    engagement_matrix['positions_seen'] = engagement_matrix['positions_seen'] / max_positions
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(engagement_matrix, annot=True, fmt='.2f', cmap='YlGnBu', cbar_kws={'label': 'Engagement Level (normalized)'}, ax=ax)
ax.set_title('Student Engagement Heatmap\n(1=Yes/Max, 0.5=Unknown, 0=No/None)', fontweight='bold', pad=20)
ax.set_xlabel('Engagement Metrics', fontweight='bold')
ax.set_ylabel('Student', fontweight='bold')
plt.tight_layout()
plt.show()

## 11. Export Cleaned Data

In [ ]:
export_df = df.copy()
export_df['companies_seen'] = export_df['companies_seen'].apply(lambda x: '; '.join(x) if isinstance(x, list) else '')
export_df['career_fields_seen'] = export_df['career_fields_seen'].apply(lambda x: '; '.join(x) if isinstance(x, list) else '')
export_cols = ['Student', 'attended_messe', 'spoke_to_people', 'positions_seen', 'companies_seen', 'career_fields_seen', 'notes']
export_df = export_df[export_cols]
export_df.to_csv(OUTPUT_CSV_PATH, index=False)
print(f"\nData exported to: {OUTPUT_CSV_PATH}")
print(f"Rows exported: {len(export_df)}")